# 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression, ElasticNetCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import json
import joblib

# 2. Загрузка файлов

In [ ]:
DATA_DIR = "../data/modeling"
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/moscow_train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/moscow_test.csv")

with open(f"{DATA_DIR}/feature_columns.json", "r", encoding="utf-8") as f:
    features = json.load(f)

with open(f"{DATA_DIR}/target_column.json", "r", encoding="utf-8") as f:
    target = json.load(f)["target"]

X_train = train_df[features].copy()
y_train = train_df[target]
y_train_exp = train_df['Price']

X_test = test_df[features].copy()
y_test = test_df[target]
y_test_exp = test_df['Price']

if "source_file" in X_train.columns:
    X_train = X_train.drop(columns=["source_file"])
    X_test = X_test.drop(columns=["source_file"])

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Числовые признаки: {len(numeric_cols)} шт.")
print(f"Категориальные признаки: {len(categorical_cols)} шт.")

Числовые признаки: 7 шт.
Категориальные признаки: 3 шт.


# 3. Функция оценки

In [ ]:
FUNCTION = "raw" # log - логарифмическая оценка, raw - обычная оценка в рублях

def evaluate(y_true, y_pred, name, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:,.4f} руб, RMSE={rmse:,.4f} руб, R²={r2:.4f}")
    return {"model": model_name, "split": name,
            "MAE": mae, "RMSE": rmse, "R2": r2}

def evaluate_log(y_true, y_pred, name, model_name):
    rmsle = np.sqrt(mean_squared_error(y_true, y_pred))
    
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon))) * 10

    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

    print(f"{name}: RMSLE={rmsle:,.4f} log, MAPE={mape:,.2f}%, sMAPE={smape:,.2f}%")
    return {"model": model_name, "split": name,"RMSLE": rmsle, "MAPE": mape, "sMAPE": smape}

# 4. Линейная регрессия

In [ ]:
MODELS = {
    "RidgeCV":            RidgeCV(alphas=np.linspace(1e-6,10,1_000_000)),
    "LassoCV":            LassoCV(alphas=np.linspace(1e-6,10,1_000_000)),
    "LinearRegressionCV": LinearRegression(),
    "ElasticNetCv": ElasticNetCV(alphas=np.linspace(1e-6,10,1_000_000))
}

all_results = []

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()) ## RobustScaler() 
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols)
    ]
)

if FUNCTION == "log":
    y_train_use = np.log1p(y_train)
    y_test_use  = np.log1p(y_test)
    eval_func = evaluate_log
else:  # FUNCTION == "raw"
    y_train_use = y_train
    y_test_use  = y_test
    eval_func = evaluate

for name, estimator in MODELS.items():
    print(f"{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model",   estimator),
    ])

    pipeline.fit(X_train, y_train_use)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    r_train = eval_func(y_train_use, y_train_pred, "Train", name)
    r_test = eval_func(y_test_use, y_test_pred, "Test", name)

    all_results.extend([r_train, r_test])

    save_path = MODELS_DIR / f"linear_{name.lower()}.joblib"
    joblib.dump(pipeline, save_path)
    print(f"  Модель сохранена: {save_path}\n")

───────────────────────────────────────────────────────
  RidgeCV
───────────────────────────────────────────────────────


KeyboardInterrupt: 

# 5. Результаты

In [ ]:
results_df = pd.DataFrame(all_results)

if (FUNCTION == "log"):
    test_summary = (results_df[results_df["split"] == "Test"]
                    .set_index("model")[["RMSLE", "MAPE", "sMAPE"]]
                    .copy())
    
    test_summary["RMSLE"]  = test_summary["RMSLE"].apply(lambda x: f"{x:.4f}")
    test_summary["MAPE"] = test_summary["MAPE"].apply(lambda x: f"{x:.2f}%")
    test_summary["sMAPE"]   = test_summary["sMAPE"].apply(lambda x: f"{x:.2f}%")
elif (FUNCTION == "raw"):
    test_summary = (results_df[results_df["split"] == "Test"]
                    .set_index("model")[["MAE", "RMSE", "R2"]]
                    .copy())
    
    test_summary["MAE"]  = test_summary["MAE"].apply(lambda x: f"{x:.3f}")
    test_summary["RMSE"] = test_summary["RMSE"].apply(lambda x: f"{x:.3f}")
    test_summary["R2"]   = test_summary["R2"].round(4)
else:
    raise ValueError("FUNCTION должна быть log или raw")

print("=" * 55)
res = "ИТОГ "
if (FUNCTION == "raw"):
    res += "(Test, исходные цены в рублях)"
elif (FUNCTION == "log"):
    res += "(Test, логарифмическая оценка)"
else:
    raise ValueError("FUNCTION должна быть log или raw")
print(res)
print("=" * 55)
print(test_summary.to_string())

ИТОГ (Test, логарифмическая оценка)
                   RMSLE   MAPE  sMAPE
model                                 
Ridge             0.3632  0.14%  1.40%
Lasso             0.3632  0.14%  1.40%
LinearRegression  0.3632  0.14%  1.40%
Elastic Net       0.3632  0.14%  1.40%
